# Pareto fronts of pose priors: sample quality against the cost of sampling (proof of concept)





The models of `posedflow_vs_rsgm.ipynb` (PoseD-Flow, DPoser, RSGM on the data of PoseD-Flow) and the HuProSO3 flow, placed
as the MLIP papers place their potentials (e.g. Fig. 1 of the GRACE paper): one panel per metric, the quality on x
(the better side on the left), the time per pose on y (log scale), and the Pareto-optimal models joined by a line.

- **Quality.** Read from the cache of `posedflow_vs_rsgm.ipynb` (`~/data/experiments/posedflow_comparison/cache`), not
  computed again:
  - **Wasserstein** (our harness): optimal transport between 2,000 generated and 2,000 test poses, degrees per joint;
  - **FID** (protocol of PoseD-Flow): Fréchet distance of the 22 SMPL joint positions against their statistics
    `gt_mean_var_v2.npz`, 20 sets of 500 samples;
  - **APD** (protocol of PoseD-Flow): mean pairwise joint distance, cm. Their table reads it as "higher is better";
    it is a diversity, so the value of real poses is drawn as a reference line.

  A model missing from that cache (the HuProSO3 flow) is evaluated here with the same settings and written to the same
  cache, so `posedflow_vs_rsgm.ipynb` reuses it.
- **Cost.** The wall-clock time of `model.sample(n)` (noise → valid rotations, all sampler steps),
  median of `REPEATS` runs after a warm-up, with the device synchronized. Two regimes, as ASE / LAMMPS in GRACE:
  - **throughput**: n = 4096 (filled markers);
  - **latency**: n = 1 (open markers).

  Checkboxes above the plots choose the n drawn; a Pareto front is drawn for each.

  The times are cached next to the metrics, one file per chip and device
  (`runtime_<chip>_<device>.json`), with the chip, device, platform and torch version in the settings. The file
  keeps every n timed so far (`BATCHES` adds the missing ones); a file of another torch is set aside as
  `runtime_<chip>_<device>.torch_<version>.json`, not overwritten. Times of other machines (e.g. a GPU server, with
  `scripts/evaluate_pareto.py`) are read from the cache: a selector above the plots chooses the measurement, and
  the plots name the machine, torch, n and repeats.
- **Shared code.** The models (`MODELS`), the settings, the cache and the timing are in `hp.evaluation.pareto`;
  `scripts/evaluate_pareto.py` evaluates and times the same models on another machine (e.g. a GPU server) into the same
  cache. Edit `MODELS` there, and choose here which of them to use.
- **† = trained on our AMASS** (SMPL-X G), evaluated on the data of PoseD-Flow: a data shift that the others do not have.
  The HuProSO3 flow is the only such model here. It is drawn as a reference for the speed of a one-pass sampler (a
  diamond), and is left out of the Pareto fronts.
- **New samplers of RSGM.** The RSGM models on the fronts (512x4 c10 with 3 seeds, 512x4 full, 1536x6) also appear with the
  samplers chosen in `rsgm_samplers.ipynb`, on the same weights and at 50 network calls: the probability flow ODE on the
  uniform grid (Euler, 50 steps) and with the EDM schedule (Heun, 25 steps). See `RSGM_SAMPLERS` in
  `hp.evaluation.pareto`. Their log-likelihood is not computed: it does not depend on the sampler.

In [ ]:
import contextlib
import html
import io
import platform
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import torch

from hp.evaluation import pareto as P
from hp.evaluation.data import load_reference

assert sys.modules["hp.evaluation.pareto"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.evaluation.pareto'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
# The models, the data and the evaluation settings are those of hp.evaluation.pareto (shared with
# scripts/evaluate_pareto.py, and the settings of posedflow_vs_rsgm.ipynb): edit MODELS there.
MODELS = P.select(None)  # all; or e.g. P.select(["RSGM", "DPoser"]) (names, short labels, or parts of them)

# Drawn, but left out of the Pareto fronts: trained on other data than the models it would be compared with
FRONT_EXCLUDE = {"HuProSO3 flow: our AMASS"}
SHOW_SAMPLERS = {"default", "heun", "euler"}  # the samplers drawn at first: each model's own, and RSGM's new ones
FRONT_STYLE = "steps"  # the fronts drawn at first: "steps" (the best quality within a time budget), "lines", "points"
TIME_TOLERANCE = 0.1  # times closer than this (relative) are a tie: of those only the best quality is on the front
SHOW_SEEDS = False  # at first a point for each seed of a model, instead of their mean (a checkbox above the plots)

# the families, in a fixed order of colours; RSGM (ours) in red, as GRACE in its figure
FAMILIES = {"RSGM": "#e34948", "PoseD-Flow": "#2a78d6", "DPoser": "#7b3fa0", "HuProSO3": "#1baf7a"}
# the families drawn at first (a checkbox each above the plots); HuProSO3 (only the reference flow †) stretches the axes
SHOW_FAMILIES = {"RSGM", "PoseD-Flow", "DPoser"}

DEVICE = P.default_device()
BATCHES = P.BATCHES  # the n to time (e.g. (1, 64, 1024, 4096, 16384)); the cache keeps every n timed before
SHOW_BATCHES = P.BATCHES  # the n checked at first in the plots (the checkboxes offer every n of a measurement)
REPEATS = P.REPEATS  # timed runs per n
MEASURE_HERE = True  # time the models on this machine (only what is not in the cache yet); False: only read the cache
FIGURES = P.CACHE.parent / "figures"  # the default folder of the save button
RECOMPUTE = set()  # names of models (or "<name> / <run>") to evaluate again (Wasserstein, FID, APD); True: all
RECOMPUTE_RUNTIME = set()  # names of models (or "<name> / <run>") to time the n of BATCHES again; True: all
# --------------------------------------------------------------------------------------
print(f"device {DEVICE}, {platform.platform()}, torch {torch.__version__}, cache {P.CACHE}")

## 1. The models and the cache

The helpers of `hp.evaluation.pareto` (as in `posedflow_vs_rsgm.ipynb`): a model with several runs (seeds) is evaluated
per run; a result is reused only if it was computed with the same settings. A run that is not on this machine is
skipped.

In [ ]:
zoo = P.Zoo(MODELS, DEVICE)
builders, identities, groups, group_of = zoo.builders, zoo.identities, zoo.groups, zoo.group_of

## 2. Quality: from the cache, or evaluated with the settings of `posedflow_vs_rsgm.ipynb`

The data is loaded as there (it sets the sizes in the cache settings). Only a model without a cached result is
evaluated (RSGM with the new samplers without the log-likelihood, `likelihood=False` in `MODELS`: it is the same as with the
default sampler and is the slow part of the harness).

In [ ]:
test = load_reference("test", P.ROTATIONS, "posedflow")
train = load_reference("train", P.ROTATIONS, "posedflow")
print(f"test {len(test):,}   train {len(train):,}")

harness, paper = P.evaluate_quality(zoo, test, train, RECOMPUTE)

## 3. Cost: the time of `model.sample(n)`


One run per model is timed (the seeds of a model have the same network and sampler). Batch 1 of a 1000-step sampler
takes 1000 network calls of a single pose: latency, not throughput. The times of this machine are measured here (what
the cache lacks); those of other machines are read from the cache.

In [ ]:
if MEASURE_HERE:
    print(f"timing on {P.chip_name(DEVICE)} ({DEVICE}), saved as {P.runtime_stage(DEVICE)}.json")
    P.measure_runtime(zoo, RECOMPUTE_RUNTIME, BATCHES, REPEATS)

MEASUREMENTS = P.runtime_stages(zoo)  # {stage: (runtime[name], info)}: every machine in the cache
for stage, (rt, info) in MEASUREMENTS.items():
    print(f"{stage}: {len(rt)} of {len(groups)} models, torch {', '.join(info['torch'])}, repeats {', '.join(info['repeats'])}"
          + (f", measured {' to '.join(dict.fromkeys(info['dates']))}" if info["dates"] else ""))

## 4. The table

Of one measurement, at the n of `SHOW_BATCHES` (the selector in section 5 shows the table of its choice too).

In [ ]:
def family(name):
    return next(f for f in FAMILIES if name.startswith(f))


def mean_over_runs(name, get):
    return float(np.mean([get(r) for r in (results[l] for l in groups[name])]))


results = {l: {"harness": harness[l], "paper": paper[l]} for l in builders}
METRICS = {  # name: (value of one run's results, unit, lower is better)
    "Wasserstein": (lambda r: r["harness"]["wasserstein"]["generated"]["joint_mean_deg"]["mean"], "deg / joint", True),
    "FID": (lambda r: r["paper"]["FID"]["mean"], "", True),
    "APD": (lambda r: r["paper"]["APD"]["mean"], "cm", False),
}
REAL = {  # real train poses, the yardstick of each metric
    "Wasserstein": next(iter(harness.values()))["wasserstein"]["contrast"]["joint_mean_deg"]["mean"],
    "FID": paper["real train poses"]["FID"]["mean"],
    "APD": paper["real train poses"]["APD"]["mean"],
}


def machine(info):
    """The machine and the settings of a measurement, in one line."""
    platform_ = ", ".join(dict.fromkeys(p.split("-")[0] + (" x86_64" if "x86_64" in p else " arm64" if "arm64" in p else "")
                                        for p in info["platform"]))
    when = f", measured {' to '.join(dict.fromkeys(info['dates']))}" if info["dates"] else ""
    return (f"{', '.join(info['chip'])} ({', '.join(info['device'])}), {platform_}, torch {', '.join(info['torch'])}; "
            f"median of {', '.join(info['repeats'])} runs after a warm-up{when}")


def timed_batches(runtime):
    """Every n of a measurement, the largest first."""
    return sorted({int(n) for rt in runtime.values() for n in rt["batches"]}, reverse=True)


def make_table(runtime, batches):
    """The rows of the models of `runtime`: times[n] in us per pose for the n of `batches` they are timed at."""
    table = []
    for name in groups:
        rt = runtime.get(name)
        if rt is None:
            print(f"{name}: not in this measurement, left out")
            continue
        times = {n: rt["batches"][str(n)]["us_per_pose"] for n in batches if str(n) in rt["batches"]}
        table.append(dict(name=name, short=MODELS[name]["short"], family=family(name), runs=len(groups[name]),
                          params_M=rt["parameters"] / 1e6, steps=rt["sample_steps"], times=times,
                          seeds=[{m: get(results[l]) for m, (get, _, _) in METRICS.items()} for l in groups[name]],
                          **{m: mean_over_runs(name, get) for m, (get, _, _) in METRICS.items()}))
    return table


def print_table(table, batches, info):
    columns = "".join(f" {f'n={n}':>10s}" for n in batches)
    print(f"{'model':24s} {'params M':>9s} {'steps':>6s}{columns} {'Wass.':>7s} {'FID':>7s} {'APD':>6s}")
    first = batches[0] if batches else None
    for r in sorted(table, key=lambda r: r["times"].get(first, np.inf)):
        times = "".join(f" {r['times'][n]:10.1f}" if n in r["times"] else f" {'-':>10s}" for n in batches)
        print(f"{r['short']:24s} {r['params_M']:9.2f} {str(r['steps']):>6s}{times} "
              f"{r['Wasserstein']:7.2f} {r['FID']:7.4f} {r['APD']:6.2f}")
    print(f"{'real train poses':24s} {'':9s} {'':6s}{' ' * 11 * len(batches)} {REAL['Wasserstein']:7.2f} {REAL['FID']:7.4f} {REAL['APD']:6.2f}")
    print(f"n=...: us per pose of sample(n). {machine(info)}.")


# the table of this machine if it is measured, else of the measurement with the most models
DEFAULT_MEASUREMENT = P.runtime_stage(DEVICE) if P.runtime_stage(DEVICE) in MEASUREMENTS else next(iter(MEASUREMENTS))
_runtime, _info = MEASUREMENTS[DEFAULT_MEASUREMENT]
_batches = [n for n in timed_batches(_runtime) if n in SHOW_BATCHES]
print_table(make_table(_runtime, _batches), _batches, _info)

## 5. The Pareto fronts

A model is Pareto-optimal if no other model is both faster and at least as good on the metric. Above the plots: the
measurement (machine), and checkboxes for the n of `sample(n)` to draw (the largest n filled markers, the others open);
a front is drawn for each n. The models of `FRONT_EXCLUDE` (diamonds) are drawn but do not take part in the fronts.
RSGM with the new samplers: stars (Heun + EDM) and crosses (ODE Euler). Checkboxes choose the samplers drawn (each model's own, and the new ones
of RSGM; the fronts are of the models drawn). A checkbox shows or hides the models of
`FRONT_EXCLUDE`, another draws a point for each seed of a model instead of their mean (the seeds share the time of
the one that is timed, and the fronts stay on the means, so that they do not pick the luckiest seed). The labels are placed so as not to overlap, with a thin line to the point when moved away from it.
The button saves the figure, on a white background, to the path in the field (the format by the extension: .png,
.pdf, .svg).

In [ ]:
def pareto(rows, metric, lower_better, n, tolerance=0.0):
    """The rows timed at n on the front, from the fastest: those no other row beats on `metric` at a time at most
    (1 + tolerance) times theirs (a tie of times within the noise of the timing; at an equal metric the faster wins)."""
    sign = 1 if lower_better else -1
    timed = [r for r in rows if n in r["times"]]

    def beats(a, r):
        if a is r or a["times"][n] > r["times"][n] * (1 + tolerance):
            return False
        return sign * a[metric] < sign * r[metric] or (a[metric] == r[metric] and a["times"][n] < r["times"][n])

    return sorted((r for r in timed if not any(beats(a, r) for a in timed)), key=lambda r: r["times"][n])


FRONT_STYLES = {"steps": "steps: the best within a time budget", "lines": "lines between the front points",
                "points": "points only: a ring on each front point"}
TOLERANCES = (0.0, 0.05, 0.1, 0.2)


def draw_front(ax, front, metric, n, k, style):
    """One front: a staircase (from each point up to the time of the next, then across to it), straight lines, or rings."""
    x, y = [r[metric] for r in front], [r["times"][n] for r in front]
    if style == "points":
        ax.plot(x, y, marker="o", ms=15 + 5 * k, ls="", mfc="none", mec=FRONT_COLOR, mew=1, zorder=1)
        return
    if style == "steps":
        x, y = [v for a in x for v in (a, a)][:-1], [y[0]] + [v for b in y[1:] for v in (b, b)]
    ax.plot(x, y, ls=DASHES[k % len(DASHES)], lw=1.4, color=FRONT_COLOR, zorder=1)


FILL = ["full", "none", "bottom", "left", "right", "top"]  # the marker of the k-th largest n
DASHES = ["--", ":", "-.", (0, (5, 2, 1, 2)), (0, (1, 3))]  # the front of the k-th largest n
FRONT_COLOR = "0.25"
SAMPLER_MARKERS = {"heun": ("*", "Heun + EDM, 50 NFE"), "euler": ("X", "ODE Euler, 50 NFE")}  # RSGM with the new samplers


SAMPLER_KINDS = {"default": "own samplers (circles)", "heun": "Heun + EDM (stars)", "euler": "ODE Euler (crosses)"}


def sampler_of(name):
    """The kind of sampler of a model: "default" (its own), or one of the new samplers of RSGM."""
    return MODELS[name].get("sampler", "default")


def marker_of(r):
    if r["name"] in FRONT_EXCLUDE:
        return "D"
    return SAMPLER_MARKERS.get(MODELS[r["name"]].get("sampler"), ("o",))[0]


def overlap(a, b):
    """The area shared by two boxes (x0, y0, x1, y1)."""
    return max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))


def place_labels(fig, labels):
    """Put the labels {ax: [(x, y, text)]} near their points, greedily: the most crowded points first, each label at the
    nearest place free of the other labels and of the markers, joined to its point by a thin line if it is far."""
    renderer, px = fig.canvas.get_renderer(), fig.dpi / 72  # pixels per point
    style = dict(fontsize=7.5, color="0.15", textcoords="offset points", ha="left", va="bottom")
    for ax, items in labels.items():
        frame = ax.get_window_extent(renderer).extents
        points = [ax.transData.transform((x, y)) for x, y, _ in items]
        markers = [(p[0] - 5 * px, p[1] - 5 * px, p[0] + 5 * px, p[1] + 5 * px) for p in points]
        taken = [t.get_window_extent(renderer).extents for t in ax.texts]  # "real train poses"
        strokes = []  # points along the drawn lines (the fronts), every 3 pixels: a label should not cross them
        for line in ax.get_lines():
            if line.get_linestyle() in ("None", "", " "):
                continue
            xy = ax.transData.transform(np.column_stack(line.get_data()))
            for a, b in zip(xy[:-1], xy[1:]):
                steps = max(1, int(np.hypot(*(b - a)) / (3 * px)))
                strokes += [a + (b - a) * s for s in np.linspace(0, 1, steps + 1)]
        strokes = np.array(strokes).reshape(-1, 2)
        crowd = [sum(np.hypot(*(p - q)) < 40 * px for q in points) for p in points]
        for i in sorted(range(len(items)), key=lambda i: -crowd[i]):
            x, y, text = items[i]
            if text is None:  # a point without a label: only in the way of the others
                continue
            probe = ax.annotate(text, (x, y), xytext=(0, 0), **style)
            box = probe.get_window_extent(renderer)
            width, height = box.width, box.height
            probe.remove()
            best = None
            for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100, 120):
                for angle in np.deg2rad(np.arange(0, 360, 15)):
                    cos, sin = np.cos(angle), np.sin(angle)
                    x0 = points[i][0] + radius * px * cos + (0 if cos > 0.3 else -width if cos < -0.3 else -width / 2)
                    y0 = points[i][1] + radius * px * sin + (0 if sin > 0.3 else -height if sin < -0.3 else -height / 2)
                    candidate = (x0, y0, x0 + width, y0 + height)
                    inside = candidate[0] >= frame[0] and candidate[1] >= frame[1] and candidate[2] <= frame[2] and candidate[3] <= frame[3]
                    area = width * height
                    cost = radius + 2000 * sum(overlap(candidate, t) for t in taken) / area \
                        + 500 * sum(overlap(candidate, m) for m in markers) / area + (0 if inside else 5000) \
                        + 40 * np.sum((strokes[:, 0] > candidate[0]) & (strokes[:, 0] < candidate[2])
                                      & (strokes[:, 1] > candidate[1]) & (strokes[:, 1] < candidate[3]))
                    if best is None or cost < best[0]:
                        best = (cost, candidate, radius)
            _, candidate, radius = best
            offset = ((candidate[0] - points[i][0]) / px, (candidate[1] - points[i][1]) / px)
            line = dict(arrowprops=dict(arrowstyle="-", color="0.6", lw=0.6, shrinkA=0, shrinkB=3)) if radius > 16 else {}
            ax.annotate(text, (x, y), xytext=offset, zorder=4, **style, **line)
            taken.append(candidate)


def figure(stage, batches, families=tuple(FAMILIES), seeds=False, samplers=tuple(SAMPLER_KINDS), style=FRONT_STYLE,
           tolerance=TIME_TOLERANCE):
    """The Pareto plots of one measurement at the n of `batches` (the largest first), on white; `families`: those drawn
    (the fronts are of the drawn models, less those of FRONT_EXCLUDE); `seeds`: a point for each seed of a model instead of their mean (the fronts stay on the means);
    `samplers`: the kinds of sampler drawn (see SAMPLER_KINDS); `style`: of the fronts (see FRONT_STYLES); `tolerance`:
    the relative tie of times (see pareto). Returns the figure, the rows and the points drawn as (artist, x, y, name), x
    and y in pixels of the figure from its top left (see highlight)."""
    runtime, info = MEASUREMENTS[stage]
    rows = [r for r in make_table(runtime, batches)
            if r["family"] in families and sampler_of(r["name"]) in samplers]
    front_rows = [r for r in rows if r["name"] not in FRONT_EXCLUDE]
    drawn = []  # (artist, x, y, name) of every point, for the clicks and the highlight
    with plt.style.context("default"):  # dark or light theme of the notebook: the figure stays white
        fig, axes = plt.subplots(1, len(METRICS), figsize=(6.5 * len(METRICS), 6.5), sharey=True)
        labels = {ax: [] for ax in axes}
        for ax, (metric, (_, unit, lower_better)) in zip(axes, METRICS.items()):
            ax.axvline(REAL[metric], color="0.6", lw=1, ls=":", zorder=0)
            ax.annotate("real train poses", (REAL[metric], 1), xycoords=("data", "axes fraction"), xytext=(3, -12),
                        textcoords="offset points", fontsize=8, color="0.4")
            for k, n in enumerate(batches):
                draw_front(ax, pareto(front_rows, metric, lower_better, n, tolerance), metric, n, k, style)
                for r in rows:
                    if n not in r["times"]:
                        continue
                    c, marker = FAMILIES[r["family"]], marker_of(r)
                    points = r["seeds"] if seeds else [r]
                    for point in points:
                        artist, = ax.plot(point[metric], r["times"][n], marker=marker, ms=(8 if k == 0 else 7) + {"*": 7, "X": 3}.get(marker, 0),
                                ls="", color=c, fillstyle=FILL[k % len(FILL)], markerfacecoloralt="white",
                                markeredgecolor=c if k else "white", markeredgewidth=0.5 if marker in "*X" else 1.2,
                                zorder=3 if k == 0 else 2)
                        drawn.append((artist, point[metric], r["times"][n], r["name"]))
                    if k == 0:  # the labels only at the largest n: the others are above or below the same x
                        label = r["short"].replace(", ODE Euler 50", " Euler").replace(", Heun EDM 50 NFE", " Heun")
                        labels[ax] += [(p[metric], r["times"][n], label if j == 0 else None) for j, p in enumerate(points)]
            ax.set_yscale("log")
            if not lower_better:
                ax.invert_xaxis()  # the better side on the left, as GRACE
            ax.set_xlabel(f"← {metric}" + (f" ({unit})" if unit else "") + ("" if lower_better else ", higher is better"))
            ax.grid(True, which="major", color="0.92", lw=0.8)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
        axes[0].set_ylabel("↓ time per pose (µs)")
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, markeredgecolor="white", markersize=8, label=f)
                   for f, c in FAMILIES.items() if any(r["family"] == f for r in rows)]
        handles += [plt.Line2D([], [], marker="o", ls="", color="0.4", fillstyle=FILL[k % len(FILL)], markerfacecoloralt="white",
                               markersize=8, label=f"n = {n}") for k, n in enumerate(batches)]
        if any(r["name"] in FRONT_EXCLUDE for r in rows):
            handles += [plt.Line2D([], [], marker="D", ls="", color="0.4", markersize=7, label="not in the fronts (†)")]
        if "default" in samplers:
            handles += [plt.Line2D([], [], marker="o", ls="", color="0.4", markeredgecolor="white", markersize=8,
                                   label="own sampler of each model")]
        handles += [plt.Line2D([], [], marker=m, ls="", color="0.4", markersize=14 if m == "*" else 10, markeredgecolor="white", markeredgewidth=0.5, label=label)
                    for kind, (m, label) in SAMPLER_MARKERS.items() if kind in samplers]
        if style == "points":
            handles += [plt.Line2D([], [], marker="o", ms=11, ls="", mfc="none", mec=FRONT_COLOR, label="on a Pareto front")]
        else:
            handles += [plt.Line2D([], [], ls=DASHES[k % len(DASHES)], color=FRONT_COLOR, label=f"Pareto front, n = {n}")
                        for k, n in enumerate(batches)]
        fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 10), frameon=False, fontsize=9)
        fig.suptitle("Sample quality against the cost of sampling; † trained on our AMASS", fontsize=11, y=0.995)
        fig.text(0.5, 0.945, machine(info) + (f"; times within {tolerance:.0%} are a tie on the fronts" if tolerance else ""), ha="center", fontsize=8.5, color="0.3")
        fig.tight_layout(rect=(0, 0.08 if len(handles) > 10 else 0.06, 1, 0.94))
        fig.patch.set_facecolor("white")
        place_labels(fig, labels)
    height = fig.bbox.height
    hits = [(a, p[0], height - p[1], name) for a, x, y, name in drawn for p in [a.axes.transData.transform((x, y))]]
    return fig, rows, hits


def highlight(hits, name):
    """Ring the points of the model `name` on every plot and fade the others (name None: all as drawn); only changes
    the artists, so a click does not lay out the figure again. Returns the rings, to remove them at the next call."""
    rings = []
    for artist, x, y, point_name in hits:
        artist.set_alpha(1 if name is None or point_name == name else 0.25)
        if point_name == name:
            rings += artist.axes.plot(*artist.get_data(), marker="o", ms=22, ls="", mfc="none", mec="black", mew=2, zorder=5)
    return rings


def default_path(stage, batches):
    return str(FIGURES / f"pareto_{stage}_n{'_'.join(map(str, batches))}.png")


# ---- the controls: measurement, n, save ----
measurement = widgets.Dropdown(
    options=[(f"{', '.join(info['chip'])} ({', '.join(info['device'])}): {len(rt)} models", stage)
             for stage, (rt, info) in MEASUREMENTS.items()],
    value=DEFAULT_MEASUREMENT, description="measurement", layout=widgets.Layout(width="520px"),
    style={"description_width": "initial"})
batch_boxes = widgets.HBox()
sampler_boxes = []
for kind, description in SAMPLER_KINDS.items():
    box = widgets.Checkbox(value=kind in SHOW_SAMPLERS, description=description, indent=False, layout=widgets.Layout(width="210px"))
    box.kind = kind
    sampler_boxes.append(box)
seeds_box = widgets.Checkbox(value=SHOW_SEEDS, description="each seed (the fronts stay on the means)", indent=False,
                             layout=widgets.Layout(width="320px"))
family_boxes = []
for family_name in FAMILIES:
    box = widgets.Checkbox(value=family_name in SHOW_FAMILIES, description=family_name, indent=False, layout=widgets.Layout(width="130px"))
    box.family = family_name
    family_boxes.append(box)
front_style = widgets.Dropdown(options=[(d, s) for s, d in FRONT_STYLES.items()], value=FRONT_STYLE, description="fronts",
                               layout=widgets.Layout(width="360px"))
tolerance_box = widgets.Dropdown(options=[(f"{t:.0%}", t) for t in sorted({*TOLERANCES, TIME_TOLERANCE})], value=TIME_TOLERANCE,
                                 description="time tie", layout=widgets.Layout(width="200px"))
path_field = widgets.Text(description="save to", layout=widgets.Layout(width="760px"), style={"description_width": "initial"})
save_button = widgets.Button(description="save", icon="save", layout=widgets.Layout(width="90px"))
status = widgets.Label()
# the plot as a picture and the text as HTML, not a widgets.Output: output printed while the cell runs can go around an
# Output (in PyCharm), stay under the controls and not be cleared by the next choice
image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
text = widgets.HTML()
picked = widgets.Label()
state = {"fig": None, "default": None, "hits": [], "highlight": None, "rings": []}


def checked():
    return [box.n for box in batch_boxes.children if box.value]


def redraw(*_):
    stage, batches = measurement.value, checked()
    if path_field.value in ("", state["default"]):  # follow the choice, unless a path was typed
        path_field.value = state["default"] = default_path(stage, batches)
    status.value = ""
    runtime, info = MEASUREMENTS[stage]
    if not batches:
        image.value, text.value, state["fig"] = b"", "<pre>no n checked</pre>", None
        return
    fig, rows, hits = figure(stage, batches, [box.family for box in family_boxes if box.value], seeds_box.value,
                             [box.kind for box in sampler_boxes if box.value], front_style.value, tolerance_box.value)
    plt.close(fig)  # kept in state for the clicks and the save button, not shown by pyplot
    state["fig"], state["hits"], state["rings"] = fig, hits, []
    show_highlight()
    lines = io.StringIO()
    with contextlib.redirect_stdout(lines):
        print_table(rows, batches, info)
        print()
        for n in batches:
            for metric, (_, _, lower_better) in METRICS.items():
                print(f"n = {n:<5d} {metric:12s} front: " + " → ".join(r["short"] for r in pareto(
                    [r for r in rows if r["name"] not in FRONT_EXCLUDE], metric, lower_better, n, tolerance_box.value)))
    text.value = f"<pre style='font-size: 12px; line-height: 1.3'>{html.escape(lines.getvalue())}</pre>"


def show_highlight():
    """Draw the highlight of state["highlight"] on the current figure and show it as the picture."""
    fig, hits = state["fig"], state["hits"]
    for ring in state["rings"]:
        ring.remove()
    if state["highlight"] not in {name for *_, name in hits}:
        state["highlight"] = None
    name = state["highlight"]
    state["rings"] = highlight(hits, name)
    picture = io.BytesIO()
    fig.savefig(picture, format="png", dpi=100, facecolor="white")
    image.value = picture.getvalue()
    picked.value = (f"highlighted: {MODELS[name]['short']} ({name}); click it again or an empty place to clear" if name
                    else "click a point to highlight its model on every plot" if CLICKS else "")


def set_batches(*_):
    """Checkboxes for the n of the chosen measurement; those of SHOW_BATCHES checked (else the largest)."""
    available = timed_batches(MEASUREMENTS[measurement.value][0])
    chosen = [n for n in available if n in SHOW_BATCHES] or available[:1]
    boxes = []
    for n in available:
        box = widgets.Checkbox(value=n in chosen, description=f"n = {n}", indent=False, layout=widgets.Layout(width="110px"))
        box.n = n
        box.observe(redraw, names="value")
        boxes.append(box)
    batch_boxes.children = boxes
    redraw()


def save(_):
    if state["fig"] is None:
        status.value = "nothing to save"
        return
    target = Path(path_field.value).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    state["fig"].savefig(target, dpi=200, facecolor="white", bbox_inches="tight")
    status.value = f"saved {target}"


def on_click(event):
    """A click on the picture: highlight the model of the nearest point (within 12 pixels), or clear the highlight."""
    x, y = event.get("dataX"), event.get("dataY")  # pixels of the picture, from its top left
    if x is None or not state["hits"]:
        return
    scale = 100 / state["fig"].dpi  # from the pixels of the figure to those of the picture (saved at dpi 100)
    distance, name = min((np.hypot(hx * scale - x, hy * scale - y), name) for _, hx, hy, name in state["hits"])
    state["highlight"] = name if distance < 12 and name != state["highlight"] else None
    show_highlight()


# the clicks need ipyevents in the kernel and in the Jupyter that shows the notebook (e.g. JupyterLab; not PyCharm)
try:
    from ipyevents import Event
    CLICKS = Event(source=image, watched_events=["click"])
    CLICKS.on_dom_event(on_click)
except ImportError:
    CLICKS = None

measurement.observe(set_batches, names="value")
seeds_box.observe(redraw, names="value")
for box in sampler_boxes + family_boxes + [front_style, tolerance_box]:
    box.observe(redraw, names="value")
save_button.on_click(save)
display(widgets.VBox([measurement, batch_boxes, widgets.HBox(family_boxes), widgets.HBox(sampler_boxes), seeds_box, widgets.HBox([front_style, tolerance_box]), widgets.HBox([path_field, save_button, status]), picked, image, text]))
set_batches()

y## 6. Reading (proof of concept)

- Every model is one point at its own sampler: the steps are not swept yet, except RSGM 512x4 (100 / 300 / 1000) and
  DPoser (100 / 1000). A sweep of the steps (5–200) for RSGM, PoseD-Flow and DPoser would turn each into a curve.
- The differences in quality are small against the noise: the ± of the metrics (in `posedflow_vs_rsgm.ipynb`) is of
  the order of the gaps between the points. The front on one metric is therefore not robust yet.
- † models are trained on other data than they are evaluated on; their quality is a lower bound.
- The times depend on this machine; on a GPU the ratio of the flow to the ODE/SDE models may differ.